### PART 5 — Compare Multiple ML Algorithms

#### Now we improve scientifically.

#### We will compare:

#### Random Forest (baseline — already done)

#### Gradient Boosting

#### Support Vector Regression (SVR)

#### K-Nearest Neighbors (KNN)

#### Linear Regression

In [3]:
# Importing the models

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

In [6]:
# Initializing the Models

models = {
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Linear Regression": LinearRegression()
}

print("Modèles prêts à être comparés !")

Modèles prêts à être comparés !


In [14]:
# importing some librairies

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [20]:
# 1. Rechargement des fichiers sauvegardés
X_fp = np.load("X_fp.npy")
y = np.load("y.npy")

# 2. Nettoyage de sécurité (enlever les infinis)
mask = np.isfinite(y)
X_clean = X_fp[mask]
y_clean = y[mask]

# 3. Création des sets Train/Test (Définition de X_train_fp, etc.)
X_train_fp, X_test_fp, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

In [12]:
# Training & Evaluating All Models

results = {}

for name, model in models.items():
    model.fit(X_train_fp, y_train)
    preds = model.predict(X_test_fp)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    results[name] = {"R2": r2, "RMSE": rmse}
    
    print(f"{name}")
    print("R²:", round(r2, 3))
    print("RMSE:", round(rmse, 3))
    print("--------------")

Random Forest
R²: 0.569
RMSE: 0.887
--------------
Gradient Boosting
R²: 0.351
RMSE: 1.088
--------------
SVR
R²: 0.545
RMSE: 0.912
--------------
KNN
R²: 0.526
RMSE: 0.93
--------------
Linear Regression
R²: -0.434
RMSE: 1.618
--------------


In [15]:
# Comparing Results Table

results_df = pd.DataFrame(results).T
results_df.sort_values(by="R2", ascending=False)

,R2,RMSE
Random Forest,0.569170,0.886854
SVR,0.544543,0.911848
KNN,0.526403,0.929830
Gradient Boosting,0.351337,1.088200
Linear Regression,-0.433674,1.617798


In [17]:
# # improving the model
# Tuning Random Forest

from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train_fp, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
Best CV R2: 0.5559125726006898


In [18]:
# Evalution of the tuned model

best_rf = grid.best_estimator_

y_pred = best_rf.predict(X_test_fp)

print("Test R2:", r2_score(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

Test R2: 0.5691699313382177
Test RMSE: 0.8868539633467997


In [23]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Recharge du CSV de base (celui avec les SMILES et pIC50)
df = pd.read_csv("cox2_clean.csv") 

# 2. Fonction pour calculer les descripteurs physiques
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return [
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol)
        ]
    return [None] * 5

# 3. Création du DataFrame avec descripteurs
desc_list = [calculate_descriptors(s) for s in df['canonical_smiles']]
df_with_descriptors = pd.DataFrame(desc_list, columns=['MW', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA'])

# 4. Préparation des matrices X (Descripteurs) et X_fp (Fingerprints déjà chargés)
X_desc = df_with_descriptors.values
X_fp = np.load("X_fp.npy") # On recharge les fingerprints binaires
y = np.load("y.npy")

# Nettoyage des NaNs/Infinis éventuels
mask = np.isfinite(y) & ~np.isnan(X_desc).any(axis=1)
X_combined = np.hstack([X_fp[mask], X_desc[mask]])
y_clean = y[mask]

# 5. Division et Entraînement Final
X_train_comb, X_test_comb, y_train, y_test = train_test_split(
    X_combined, y_clean, test_size=0.2, random_state=42
)

rf_comb = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_comb.fit(X_train_comb, y_train)

# 6. Résultat
y_pred_comb = rf_comb.predict(X_test_comb)
print(f"Combined R2 Score : {r2_score(y_test, y_pred_comb):.3f}")


Combined R2 Score : 0.560
